In [0]:
# Workflow execution

from datetime import datetime

dbutils.widgets.text(
    "file_date",
    datetime.now().strftime("%Y%m%d")
)

file_date = dbutils.widgets.get("file_date")

print(f"Processing data for file_date = {file_date}")

In [0]:
# Manual execution

#file_date = "20260207"

#print(f"Processing data for file_date = {file_date}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

### Remove Existing Data for the Current file_date

In [0]:
silver_tables = [
    "customers_unified",
    "products_unified",
    "orders_unified"
]

for table in silver_tables:
    table_name = f"`namastesql-databricks-catalog`.silver.{table}"

    if spark.catalog.tableExists(table_name):
        spark.sql(f"""
            DELETE FROM {table_name}
            WHERE file_date = '{file_date}'
        """)

### Customer Unified Dataset

In [0]:
spark.table(
    "`namastesql-databricks-catalog`.bronze.online_customers"
).printSchema()

spark.table(
    "`namastesql-databricks-catalog`.bronze.offline_customers"
).printSchema()

In [0]:
display(
    spark.table(
        "`namastesql-databricks-catalog`.bronze.online_customers"
    ).limit(2)
)

display(
    spark.table(
        "`namastesql-databricks-catalog`.bronze.offline_customers"
    ).limit(2)
)

In [0]:
# Read online and offline customer data from Bronze layer
# Filter only the records belonging to the current file_date

online_customers = (
    spark.table("`namastesql-databricks-catalog`.bronze.online_customers")
         .filter(col("file_date") == file_date)
)


offline_customers = (
    spark.table("`namastesql-databricks-catalog`.bronze.offline_customers")
         .filter(col("file_date") == file_date)
)

In [0]:
# Standardize online customer columns
# Clean name, email, city and state
# Clean mobile number by removing special characters and keeping the last 10 digits
# Add store_id as 'NA' since online customers are not associated with a physical store

online_customers = (
    online_customers
        .select(
            col("customer_id"),
            initcap(trim(col("full_name"))).alias("customer_name"),
            lower(trim(col("email"))).alias("email"),
            regexp_replace(
                trim(col("mobile_number")),
                "[^0-9]",
                ""
            ).alias("mobile_digits"),
            initcap(trim(col("city"))).alias("city"),
            upper(trim(col("state"))).alias("state"),
            col("zipcode"),
            col("signup_date"),
            col("file_date"),
            col("source"),
            col("ingest_ts")
        )
        .withColumn(
            "mobile_clean",
            expr("right(mobile_digits, 10)")
        )
        .drop("mobile_digits")
        .withColumn("store_id", lit("NA"))
)

display(online_customers.limit(2))

In [0]:
# Standardize offline customer columns so that both sources have the same schema
# Clean name, email, city and state
# Clean mobile number by removing special characters and keeping the last 10 digits
# Add signup_date as 'NA' since offline customers do not have this attribute

offline_customers = (
    offline_customers
        .select(
            col("cust_id").alias("customer_id"),
            initcap(trim(col("name"))).alias("customer_name"),
            lower(trim(col("email"))).alias("email"),
            regexp_replace(
                trim(col("phone")),
                "[^0-9]",
                ""
            ).alias("mobile_digits"),
            initcap(trim(col("city"))).alias("city"),
            upper(trim(col("state"))).alias("state"),
            col("pincode").alias("zipcode"),
            lit("NA").alias("signup_date"),
            col("store_id"),
            col("file_date"),
            col("source"),
            col("ingest_ts")
        )
        .withColumn(
            "mobile_clean",
            expr("right(mobile_digits, 10)")
        )
        .drop("mobile_digits")
)

display(offline_customers.limit(2))

In [0]:
# Identify customers having invalid mobile numbers
# Invalid mobile numbers are those having less than 10 digits after cleaning
# Add a reject reason column for auditing purposes

customers_rejected = (
    online_customers
        .unionByName(offline_customers)
        .filter(
            col("mobile_clean").isNull() |
            (length(col("mobile_clean")) < 10)
        )
        .withColumn(
            "reject_reason",
            lit("Invalid mobile number")
        )
)

In [0]:
# Remove invalid customer records from further processing
# Only customers having exactly 10 digits in mobile_clean are retained

online_customers = (
    online_customers
        .filter(length(col("mobile_clean")) == 10)
)

offline_customers = (
    offline_customers
        .filter(length(col("mobile_clean")) == 10)
)

In [0]:
# Remove duplicate customers within the same source
# If multiple records exist for the same mobile number,
# keep the latest record based on ingest_ts

customer_window = (
    Window
        .partitionBy("mobile_clean")
        .orderBy(col("ingest_ts").desc())
)

online_customers = (
    online_customers
        .withColumn(
            "row_num",
            row_number().over(customer_window)
        )
        .filter(col("row_num") == 1)
        .drop("row_num")
)

offline_customers = (
    offline_customers
        .withColumn(
            "row_num",
            row_number().over(customer_window)
        )
        .filter(col("row_num") == 1)
        .drop("row_num")
)

In [0]:
# Perform a full outer join using mobile_clean as the business key
# Give preference to online data whenever a customer exists in both systems
# Keep offline data only when no corresponding online customer exists
# Produce one unified record per customer

customers_unified = (
    online_customers.alias("on")
        .join(
            offline_customers.alias("off"),
            col("on.mobile_clean") == col("off.mobile_clean"),
            "full_outer"
        )
        .select(
            coalesce(
                col("on.customer_id"),
                col("off.customer_id")
            ).alias("customer_id"),

            coalesce(
                col("on.customer_name"),
                col("off.customer_name")
            ).alias("customer_name"),

            coalesce(
                col("on.email"),
                col("off.email")
            ).alias("email"),

            coalesce(
                col("on.mobile_clean"),
                col("off.mobile_clean")
            ).alias("mobile_clean"),

            coalesce(
                col("on.city"),
                col("off.city")
            ).alias("city"),

            coalesce(
                col("on.state"),
                col("off.state")
            ).alias("state"),

            coalesce(
                col("on.zipcode"),
                col("off.zipcode")
            ).alias("zipcode"),

            coalesce(
                col("on.signup_date"),
                col("off.signup_date")
            ).alias("signup_date"),

            coalesce(
                col("on.store_id"),
                col("off.store_id")
            ).alias("store_id"),

            coalesce(
                col("on.file_date"),
                col("off.file_date")
            ).alias("file_date"),

            coalesce(
                col("on.source"),
                col("off.source")
            ).alias("source"),

            coalesce(
                col("on.ingest_ts"),
                col("off.ingest_ts")
            ).alias("ingest_ts")
        )
)

In [0]:
# Append the unified customer dataset to the Silver layer table
# The table will be automatically created during the first execution

customers_unified.write \
    .mode("append") \
    .saveAsTable(
        "`namastesql-databricks-catalog`.silver.customers_unified"
    )

In [0]:
# Append invalid customer records to the reject table
# This table is useful for data quality checks and debugging

customers_rejected.write \
    .mode("append") \
    .saveAsTable(
        "`namastesql-databricks-catalog`.silver.customers_rejected"
    )

In [0]:
display(customers_unified)
customers_unified.count()

### Products Unified Dataset

In [0]:
spark.table(
    "`namastesql-databricks-catalog`.bronze.online_products"
).printSchema()

spark.table(
    "`namastesql-databricks-catalog`.bronze.offline_products"
).printSchema()

In [0]:
display(
    spark.table(
        "`namastesql-databricks-catalog`.bronze.online_products"
    ).limit(2)
)

display(
    spark.table(
        "`namastesql-databricks-catalog`.bronze.offline_products"
    ).limit(2)
)

In [0]:
# Read online and offline product data from Bronze layer
# Filter only the records belonging to the current file_date

online_products = (
    spark.table("`namastesql-databricks-catalog`.bronze.online_products")
         .filter(col("file_date") == file_date)
)

display(online_products.limit(2))

offline_products = (
    spark.table("`namastesql-databricks-catalog`.bronze.offline_products")
         .filter(col("file_date") == file_date)
)

display(offline_products.limit(2))

In [0]:
# Standardize online product columns
# Clean text columns and convert price to double

online_products = (
    online_products
        .select(
            col("sku_id"),
            initcap(trim(col("product_name"))).alias("product_name"),
            initcap(trim(col("category"))).alias("category"),
            initcap(trim(col("brand"))).alias("brand"),
            col("price").cast("double").alias("price"),
            upper(trim(col("currency"))).alias("currency"),
            upper(trim(col("is_active"))).alias("is_active"),
            lit(None).cast("double").alias("tax_rate"),
            col("file_date"),
            col("source"),
            col("ingest_ts")
        )
)

In [0]:
# Standardize offline product columns so that both sources have the same schema
# Clean text columns and convert price and tax_rate to double
# Set currency to USD and is_active to Y for offline products

offline_products = (
    offline_products
        .select(
            col("sku").alias("sku_id"),
            initcap(trim(col("name"))).alias("product_name"),
            initcap(trim(col("category_name"))).alias("category"),
            initcap(trim(col("brand_name"))).alias("brand"),
            col("mrp_price").cast("double").alias("price"),
            lit("USD").alias("currency"),
            lit("Y").alias("is_active"),
            col("tax_rate").cast("double").alias("tax_rate"),
            col("file_date"),
            col("source"),
            col("ingest_ts")
        )
)

In [0]:
# Defensive data quality check
# If duplicate products exist within the same source for a SKU,
# keep the latest record based on ingest_ts

product_window = (
    Window
        .partitionBy("sku_id")
        .orderBy(col("ingest_ts").desc())
)

online_products = (
    online_products
        .withColumn(
            "row_num",
            row_number().over(product_window)
        )
        .filter(col("row_num") == 1)
        .drop("row_num")
)

offline_products = (
    offline_products
        .withColumn(
            "row_num",
            row_number().over(product_window)
        )
        .filter(col("row_num") == 1)
        .drop("row_num")
)

In [0]:
# Perform a full outer join using sku_id as the business key
# Give preference to online data whenever a product exists in both systems
# Keep offline data only when no corresponding online product exists
# Produce one unified record per product

products_unified = (
    online_products.alias("on")
        .join(
            offline_products.alias("off"),
            col("on.sku_id") == col("off.sku_id"),
            "full_outer"
        )
        .select(
            coalesce(
                col("on.sku_id"),
                col("off.sku_id")
            ).alias("sku_id"),

            coalesce(
                col("on.product_name"),
                col("off.product_name")
            ).alias("product_name"),

            coalesce(
                col("on.category"),
                col("off.category")
            ).alias("category"),

            coalesce(
                col("on.brand"),
                col("off.brand")
            ).alias("brand"),

            coalesce(
                col("on.price"),
                col("off.price")
            ).alias("price"),

            coalesce(
                col("on.currency"),
                col("off.currency")
            ).alias("currency"),

            coalesce(
                col("on.is_active"),
                col("off.is_active")
            ).alias("is_active"),

            coalesce(
                col("on.tax_rate"),
                col("off.tax_rate")
            ).alias("tax_rate"),

            coalesce(
                col("on.file_date"),
                col("off.file_date")
            ).alias("file_date"),

            coalesce(
                col("on.source"),
                col("off.source")
            ).alias("source"),

            coalesce(
                col("on.ingest_ts"),
                col("off.ingest_ts")
            ).alias("ingest_ts")
        )
)

In [0]:
display(products_unified.filter(col("source") == "offline").limit(2))

In [0]:
# Append the unified product dataset to the Silver layer table
# The table will be automatically created during the first execution

products_unified.write \
    .mode("append") \
    .saveAsTable(
        "`namastesql-databricks-catalog`.silver.products_unified"
    )

### Orders Unified Dataset

In [0]:
spark.table(
    "`namastesql-databricks-catalog`.bronze.online_orders"
).printSchema()

spark.table(
    "`namastesql-databricks-catalog`.bronze.offline_orders"
).printSchema()

In [0]:
display(
    spark.table(
        "`namastesql-databricks-catalog`.bronze.online_orders"
    ).limit(2)
)

display(
    spark.table(
        "`namastesql-databricks-catalog`.bronze.offline_orders"
    ).limit(2)
)

In [0]:
# Read online and offline order data from Bronze layer
# Filter only the records belonging to the current file_date

online_orders = (
    spark.table("`namastesql-databricks-catalog`.bronze.online_orders")
         .filter(col("file_date") == file_date)
)

offline_orders = (
    spark.table("`namastesql-databricks-catalog`.bronze.offline_orders")
         .filter(col("file_date") == file_date)
)

In [0]:
# Standardize online order columns
# Convert datatypes and add store_num as NA

online_orders = (
    online_orders
        .select(
            col("order_id"),
            col("order_ts").cast("timestamp").alias("order_ts"),
            col("customer_id"),
            col("sku_id"),
            col("qty").cast("int").alias("quantity"),
            col("list_price").cast("double").alias("list_price"),
            coalesce(
                col("discount").cast("double"),
                lit(0.0)
            ).alias("discount"),
            upper(trim(col("payment_mode"))).alias("payment_mode"),
            lit("NA").alias("store_num"),
            col("file_date"),
            col("source"),
            col("ingest_ts")
        )
        .withColumn(
            "amount",
            col("list_price") - col("discount")
        )
)

In [0]:
# Standardize offline order columns so that both sources have the same schema
# Convert datatypes and calculate final amount

offline_orders = (
    offline_orders
        .select(
            col("txn_id").alias("order_id"),
            col("txn_time").cast("timestamp").alias("order_ts"),
            col("customer_id"),
            col("sku").alias("sku_id"),
            col("quantity").cast("int").alias("quantity"),
            col("mrp").cast("double").alias("list_price"),
            coalesce(
                col("discount").cast("double"),
                lit(0.0)
            ).alias("discount"),
            upper(trim(col("payment_mode"))).alias("payment_mode"),
            col("store_num"),
            col("file_date"),
            col("source"),
            col("ingest_ts")
        )
        .withColumn(
            "amount",
            col("list_price") - col("discount")
        )
)

In [0]:
# Read online customers from Bronze layer
# Clean the mobile number and create a customer-to-mobile lookup

online_customer_lookup = (
    spark.table("`namastesql-databricks-catalog`.bronze.online_customers")
         .filter(col("file_date") == file_date)
         .select(
             col("customer_id"),
             regexp_replace(
                 trim(col("mobile_number")),
                 "[^0-9]",
                 ""
             ).alias("mobile_digits")
         )
         .withColumn(
             "mobile_clean",
             expr("right(mobile_digits, 10)")
         )
         .drop("mobile_digits")
)

In [0]:
# Read offline customers from Bronze layer
# Clean the mobile number and create a customer-to-mobile lookup

offline_customer_lookup = (
    spark.table("`namastesql-databricks-catalog`.bronze.offline_customers")
         .filter(col("file_date") == file_date)
         .select(
             col("cust_id").alias("customer_id"),
             regexp_replace(
                 trim(col("phone")),
                 "[^0-9]",
                 ""
             ).alias("mobile_digits")
         )
         .withColumn(
             "mobile_clean",
             expr("right(mobile_digits, 10)")
         )
         .drop("mobile_digits")
)

In [0]:
# Join online orders with online customer lookup
# This adds mobile_clean to every online order

online_orders = (
    online_orders
        .join(
            online_customer_lookup,
            "customer_id",
            "left"
        )
)

# Join offline orders with offline customer lookup
# This adds mobile_clean to every offline order

offline_orders = (
    offline_orders
        .join(
            offline_customer_lookup,
            "customer_id",
            "left"
        )
)

In [0]:
display(
    online_orders.select(
        "customer_id",
        "mobile_clean"
    )
)

display(
    offline_orders.select(
        "customer_id",
        "mobile_clean"
    )
)



In [0]:
online_orders.filter(
    col("mobile_clean").isNull()
).count()


In [0]:
offline_orders.filter(
    col("mobile_clean").isNull()
).count()

In [0]:
offline_orders.count()

In [0]:
offline_orders.filter(
    col("mobile_clean").isNull()
).select(
    "customer_id"
).distinct().count()

In [0]:
offline_customer_lookup.filter(
    col("mobile_clean").isNull()
).count()

In [0]:
# Combine online and offline orders
# Every order now has a mobile_clean column

orders_with_mobile = (
    online_orders
        .unionByName(offline_orders)
)

In [0]:
display(orders_with_mobile.limit(2))

In [0]:
# Check whether the same mobile number is associated with multiple customer IDs
# This justifies replacing source customer IDs with unified customer IDs

display(orders_with_mobile \
    .groupBy("mobile_clean") \
    .agg(
        countDistinct("customer_id").alias("customer_count")
    ) \
    .filter(col("customer_count") > 1))

In [0]:
# Create mobile-to-unified-customer lookup
# This will help us replace source customer IDs with the final customer ID

customer_lookup = (
    spark.table(
        "`namastesql-databricks-catalog`.silver.customers_unified"
    )
    .filter(col("file_date") == file_date)
    .select(
        "mobile_clean",
        col("customer_id").alias("final_customer_id")
    )
)

# Map every order to the final unified customer ID
# Join is performed using mobile_clean

orders_with_customer = (
    orders_with_mobile
        .join(
            customer_lookup,
            "mobile_clean",
            "left"
        )
)

In [0]:
# Check orders that could not be mapped
# These are potential reject records

orders_with_customer.filter(
    col("final_customer_id").isNull()
).count()

In [0]:
# Orders that could not be mapped to a unified customer
# These orders do not have a valid final_customer_id

orders_rejected = (
    orders_with_customer
        .filter(
            col("final_customer_id").isNull()
        )
        .withColumn(
            "reject_reason",
            lit("Customer not found in customers_unified")
        )
)

In [0]:
# Keep only valid orders
# Replace original customer_id with the final unified customer_id
# Remove helper columns used for lookup

orders_unified = (
    orders_with_customer
        .filter(
            col("final_customer_id").isNotNull()
        )
        .drop("customer_id")
        .withColumnRenamed(
            "final_customer_id",
            "customer_id"
        )
        .drop("mobile_clean")
)

In [0]:
orders_rejected.count()


In [0]:
orders_unified.count()

In [0]:
orders_rejected.count() + orders_unified.count()

In [0]:
orders_with_mobile.count()

In [0]:
# Append rejected orders to the Silver reject table
# The table will be automatically created during the first execution

orders_rejected.write \
    .mode("append") \
    .saveAsTable(
        "`namastesql-databricks-catalog`.silver.orders_rejected"
    )

In [0]:
# Append the unified order dataset to the Silver layer table
# The table will be automatically created during the first execution

orders_unified.write \
    .mode("append") \
    .saveAsTable(
        "`namastesql-databricks-catalog`.silver.orders_unified"
    )

In [0]:
# Verify that no records were lost during processing

print(f"Orders with Mobile : {orders_with_mobile.count()}")
print(f"Orders Unified     : {orders_unified.count()}")
print(f"Orders Rejected    : {orders_rejected.count()}")
print(
    f"Total Processed    : "
    f"{orders_unified.count() + orders_rejected.count()}"
)